In [1]:
import numpy as np
import pandas as pd

In [2]:
temp_df = pd.read_csv('IMDB Dataset.csv')

In [3]:
df = temp_df.iloc[:10000]

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.drop_duplicates(inplace=True)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11372\3006716147.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace=True)


In [6]:
import re
def remove_tags(raw_text):
    cleaned_text = re.sub(re.compile('<.*?>'), '', raw_text)
    return cleaned_text

In [7]:
df['review'] = df['review'].apply(remove_tags)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11372\2336150696.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(remove_tags)


In [8]:
df['review'] = df['review'].apply(lambda x:x.lower())

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11372\740760900.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(lambda x:x.lower())


In [9]:
from nltk.corpus import stopwords

sw_list = stopwords.words('english')

df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11372\2826946130.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))


In [10]:
df['review']

0       one reviewers mentioned watching 1 oz episode ...
1       wonderful little production. filming technique...
2       thought wonderful way spend time hot summer we...
3       basically there's family little boy (jake) thi...
4       petter mattei's "love time money" visually stu...
                              ...                        
9995    fun, entertaining movie wwii german spy (julie...
9996    give break. anyone say "good hockey movie"? kn...
9997    movie bad movie. watching endless series bad h...
9998    movie probably made entertain middle school, e...
9999    smashing film film-making. shows intense stran...
Name: review, Length: 9983, dtype: object

In [11]:
import gensim

In [12]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [13]:
story = []
for doc in df['review']:
    raw_sent = sent_tokenize(doc)
    for sent in raw_sent:
        story.append(simple_preprocess(sent))
    

In [14]:
model = gensim.models.Word2Vec(
    window=10,
    min_count=2
)

In [15]:
model.build_vocab(story)

In [16]:
model.train(story, total_examples=model.corpus_count, epochs=model.epochs)

(5849405, 6186875)

In [17]:
len(model.wv.index_to_key)

31845

In [18]:
def document_vector(doc):
    # remove out-of-vocabulary words
    doc = [word for word in doc.split() if word in model.wv.index_to_key]
    return np.mean(model.wv[doc], axis=0)

In [19]:
document_vector(df['review'].values[0])

array([ 0.06723595,  0.3568713 ,  0.24874228, -0.00260204, -0.10102165,
       -0.5568562 ,  0.1453251 ,  0.8315505 , -0.19378346, -0.1810615 ,
       -0.21448298, -0.5985487 , -0.06126373,  0.2670551 ,  0.23745336,
       -0.2344991 ,  0.16314186, -0.5374161 , -0.04479212, -0.6368561 ,
        0.02575962,  0.08542295,  0.21201909, -0.23690408, -0.20085181,
       -0.12866712, -0.10555297, -0.17173244, -0.28986588, -0.03646101,
        0.3631751 ,  0.03913074, -0.04775257, -0.18622655, -0.40787637,
        0.38874763,  0.06230502, -0.2972488 , -0.09804316, -0.71659166,
        0.1860467 , -0.21113871, -0.15261783, -0.01343639,  0.41668528,
       -0.13320799, -0.43914133, -0.20592089,  0.07467078,  0.3558181 ,
        0.1841564 , -0.27887064, -0.4026406 , -0.13472047, -0.3607324 ,
        0.05805083,  0.41486505,  0.20985778, -0.29855022,  0.09285806,
       -0.02588812,  0.18451142, -0.17906798,  0.05351073, -0.4020089 ,
        0.44592574,  0.01132941,  0.11643194, -0.5552275 ,  0.32

In [20]:
from tqdm import tqdm

In [21]:
X = []
for doc in tqdm(df['review'].values):
    X.append(document_vector(doc))

  0%|          | 0/9983 [00:00<?, ?it/s]

100%|██████████| 9983/9983 [06:23<00:00, 26.03it/s]


In [22]:
X = np.array(X)

In [23]:
X[0]

array([ 0.06723595,  0.3568713 ,  0.24874228, -0.00260204, -0.10102165,
       -0.5568562 ,  0.1453251 ,  0.8315505 , -0.19378346, -0.1810615 ,
       -0.21448298, -0.5985487 , -0.06126373,  0.2670551 ,  0.23745336,
       -0.2344991 ,  0.16314186, -0.5374161 , -0.04479212, -0.6368561 ,
        0.02575962,  0.08542295,  0.21201909, -0.23690408, -0.20085181,
       -0.12866712, -0.10555297, -0.17173244, -0.28986588, -0.03646101,
        0.3631751 ,  0.03913074, -0.04775257, -0.18622655, -0.40787637,
        0.38874763,  0.06230502, -0.2972488 , -0.09804316, -0.71659166,
        0.1860467 , -0.21113871, -0.15261783, -0.01343639,  0.41668528,
       -0.13320799, -0.43914133, -0.20592089,  0.07467078,  0.3558181 ,
        0.1841564 , -0.27887064, -0.4026406 , -0.13472047, -0.3607324 ,
        0.05805083,  0.41486505,  0.20985778, -0.29855022,  0.09285806,
       -0.02588812,  0.18451142, -0.17906798,  0.05351073, -0.4020089 ,
        0.44592574,  0.01132941,  0.11643194, -0.5552275 ,  0.32

In [24]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

y = encoder.fit_transform(df['sentiment'])

In [25]:
y

array([1, 1, 1, ..., 0, 0, 1])

In [26]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [28]:
rf = RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test,y_pred)

0.7666499749624437